In [ ]:
%run ./expectations

In [ ]:
dbutils.widgets.dropdown("magnitud_target", "no2", [
    "no2", "no", "nox", "pm10", "pm2_5", "o3", "so2", "co",
    "tol", "ben", "ebe", "ch4", "nmhc", "tch",
])

# Empty means the city-wide variant, not a per-district one
dbutils.widgets.text("distrito", "")

MAGNITUD_TARGET = dbutils.widgets.get("magnitud_target")
DISTRITO = dbutils.widgets.get("distrito")

if DISTRITO:
    suffix = f"_D{DISTRITO.zfill(2)}_7"
    FEATURE_TABLE = f"{ML_TABLE}.features_{MAGNITUD_TARGET}{suffix}"
    FORECAST_TABLE = f"{ML_TABLE}.forecast_{MAGNITUD_TARGET}{suffix}"
else:
    FEATURE_TABLE = f"{ML_TABLE}.features_{MAGNITUD_TARGET}"
    FORECAST_TABLE = f"{ML_TABLE}.forecast_{MAGNITUD_TARGET}_7"

NOTEBOOK = "tests/ml"
errors = []
print(f"testing {FEATURE_TABLE} and {FORECAST_TABLE}")

features = spark.table(FEATURE_TABLE)
forecast = spark.table(FORECAST_TABLE)

In [ ]:
# Unity Catalog declares the PK but never enforces it
errors += [c for c in [
    expect_unique("features_pk", features, ["cod_dis", "fecha"], NOTEBOOK),
    expect("features_keys", features, "cod_dis IS NOT NULL AND fecha IS NOT NULL", NOTEBOOK),
] if c]

if features.isEmpty():
    errors.append(error_record(NOTEBOOK, Exception("features_rows: table is empty")))

In [ ]:
errors += [c for c in [
    expect_unique("forecast_grain", forecast, ["distrito", "fecha"], NOTEBOOK),
    expect("forecast_valor", forecast,
           "valor_predicho IS NOT NULL AND valor_predicho >= 0", NOTEBOOK),
] if c]

In [ ]:
# Every district must carry the full 7 day horizon, starting after its last observation
HORIZON = 7
last_observed = features.selectExpr("max(fecha)").first()[0]

incomplete = forecast.groupBy("distrito").count().filter(f"count <> {HORIZON}").count()
print(f"expect forecast_horizon: {incomplete} districts off horizon")
if incomplete:
    errors.append(error_record(
        NOTEBOOK, Exception(f"forecast_horizon: {incomplete} districts without {HORIZON} rows")
    ))

errors += [c for c in [
    expect("forecast_fecha_futura", forecast, f"fecha > '{last_observed}'", NOTEBOOK),
] if c]

In [ ]:
report(errors, NOTEBOOK)